# HLK-ViT5

Notebook huấn luyện **ViT5-base** trên `8Opt/vietnamese-summarization-dataset-0001` với ba cải tiến:

1. **Long-context phân cấp thực dụng:** tách câu → tạo chunk chồng lấn → chấm điểm trên toàn văn → chọn Top-K chunk.
2. **Keyword-aware multi-task:** huấn luyện đồng thời tóm tắt và sinh từ khóa.
3. **Factuality reranking:** sinh nhiều ứng viên, ưu tiên bản bám sát từ khóa, thực thể và số liệu của văn bản gốc.



In [2]:
# ============================================================
# 1. CẤU HÌNH DỰ ÁN LOCAL TRONG VS CODE
# ============================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

LOCAL_MODEL = (
    PROJECT_ROOT
    / "models"
    / "ViT5-base"
)

LOCAL_DATASET = (
    PROJECT_ROOT
    / "datasets"
    / "vietnamese-summarization-dataset-0001"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "training"
    / "vit5-base-hlk-0001-checkpoints"
)

FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "ViT5-base-HLK-0001"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "vit5-base-HLK-0001"
)

for directory in [
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
    RESULT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

assert LOCAL_MODEL.exists(), (
    f"Không tìm thấy model tại: {LOCAL_MODEL}"
)

assert LOCAL_DATASET.exists(), (
    f"Không tìm thấy dataset tại: {LOCAL_DATASET}"
)

print("Python       :", sys.executable)
print("Project root :", PROJECT_ROOT)
print("Model gốc    :", LOCAL_MODEL)
print("Dataset      :", LOCAL_DATASET)
print("Checkpoint   :", CHECKPOINT_DIR)
print("Model cuối   :", FINAL_MODEL_DIR)
print("Kết quả      :", RESULT_DIR)


Python       : d:\homework\BTL_NLP\.venv\Scripts\python.exe
Project root : D:\homework\BTL_NLP
Model gốc    : D:\homework\BTL_NLP\models\ViT5-base
Dataset      : D:\homework\BTL_NLP\datasets\vietnamese-summarization-dataset-0001
Checkpoint   : D:\homework\BTL_NLP\training\vit5-base-hlk-0001-checkpoints
Model cuối   : D:\homework\BTL_NLP\models\ViT5-base-HLK-0001
Kết quả      : D:\homework\BTL_NLP\results\vit5-base-HLK-0001


In [3]:
# ============================================================
# 2. CÀI THƯ VIỆN
# ============================================================

# Chạy một lần nếu môi trường chưa cài đủ.
# Không đặt torch/torchvision/torchaudio ở đây để tránh ghi đè
# bản PyTorch CUDA đã cài riêng.
#
# %pip install -U transformers datasets accelerate evaluate \
#     rouge-score sentencepiece protobuf scikit-learn


In [4]:
# ============================================================
# 3. KIỂM TRA GPU VÀ LOAD DATASET
# ============================================================

import torch
from datasets import DatasetDict, load_dataset

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )
else:
    print("Cảnh báo: đang chạy bằng CPU.")

data_files = {
    "train": str(
        LOCAL_DATASET / "train.jsonl"
    ),
    "validation": str(
        LOCAL_DATASET / "validation.jsonl"
    ),
    "test": str(
        LOCAL_DATASET / "test.jsonl"
    ),
}

raw_dataset = load_dataset(
    "json",
    data_files=data_files,
)

print(raw_dataset)
print(
    "Các cột:",
    raw_dataset["train"].column_names,
)
print(
    "Mẫu đầu tiên:",
    raw_dataset["train"][0],
)









import sys
import torch

print("Python executable :", sys.executable)
print("Python version    :", sys.version)
print("Torch version     :", torch.__version__)
print("Torch location    :", torch.__file__)
print("CUDA build        :", torch.version.cuda)
print("CUDA available    :", torch.cuda.is_available())
print("cuDNN version     :", torch.backends.cudnn.version())

if torch.cuda.is_available():
    print("GPU name          :", torch.cuda.get_device_name(0))
    print(
        "GPU VRAM         :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )

CUDA: True
GPU : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB
DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 15620
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1953
    })
})
Các cột: ['document', 'summary', 'keywords']
Mẫu đầu tiên: {'document': 'Lá N của cây N lô hội N chứa V đầy A chất N gel N và bạn N có thể hái V mỗi khi N cần V . Nên V để khi N nào dùng V mới hái V . Cắt N một nhánh N từ cây lô hội N và vắt V hoặc múc V phần N gel V trong suốt A bên N trong N ra V . Nếu N thu hoạch V nhiều A , bạn N có thể cắt V đôi lá N cây N lô hội N ( theo V chiều dọc N ) để lấy V được hết V chất N gel V bên N trong N . Chỉ N nên hái V đủ A cho mỗi lần N sử dụng V . Nếu N còn thừa A , bạn N có thể đựng V trong hộp N có V nắp N đậy N kín A và trữ V trong tủ lạnh N được

In [5]:
# ============================================================
# 4. LÀM SẠCH, DEBUG VÀ QUẢN LÝ CHECKPOINT
# ============================================================

TEXT_COLUMN = "document"
SUMMARY_COLUMN = "summary"
KEYWORDS_COLUMN = "keywords"


def valid_example(example):
    document = example.get(TEXT_COLUMN)
    summary = example.get(SUMMARY_COLUMN)

    return (
        isinstance(document, str)
        and isinstance(summary, str)
        and bool(document.strip())
        and bool(summary.strip())
    )


clean_dataset = DatasetDict()

for split_name, split_dataset in raw_dataset.items():
    before = len(split_dataset)

    filtered = split_dataset.filter(
        valid_example,
        desc=f"Cleaning {split_name}",
    )

    clean_dataset[split_name] = filtered

    print(
        f"{split_name}: "
        f"{len(filtered):,}/{before:,} mẫu hợp lệ"
    )

raw_dataset = clean_dataset


DEBUG_MODE = False
DEBUG_TRAIN_SIZE = 500
DEBUG_VALIDATION_SIZE = 100
DEBUG_TEST_SIZE = 100

if DEBUG_MODE:
    dataset_for_training = DatasetDict({
        "train": raw_dataset["train"].select(
            range(
                min(
                    DEBUG_TRAIN_SIZE,
                    len(raw_dataset["train"]),
                )
            )
        ),
        "validation": raw_dataset[
            "validation"
        ].select(
            range(
                min(
                    DEBUG_VALIDATION_SIZE,
                    len(
                        raw_dataset["validation"]
                    ),
                )
            )
        ),
        "test": raw_dataset["test"].select(
            range(
                min(
                    DEBUG_TEST_SIZE,
                    len(raw_dataset["test"]),
                )
            )
        ),
    })

    print("Đang chạy chế độ DEBUG.")
else:
    dataset_for_training = raw_dataset


# Chỉ đổi thành True khi chủ động train lại từ đầu.
RESET_CHECKPOINTS = False

if RESET_CHECKPOINTS:
    import shutil

    if CHECKPOINT_DIR.exists():
        shutil.rmtree(CHECKPOINT_DIR)

    CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Đã chủ động xóa checkpoint.")
else:
    checkpoint_names = sorted(
        path.name
        for path in CHECKPOINT_DIR.glob(
            "checkpoint-*"
        )
        if path.is_dir()
    )

    print("Giữ nguyên checkpoint hiện có.")
    print(
        "Checkpoint:",
        checkpoint_names
        if checkpoint_names
        else "chưa có",
    )

print(dataset_for_training)


train: 15,620/15,620 mẫu hợp lệ
validation: 1,952/1,952 mẫu hợp lệ
test: 1,953/1,953 mẫu hợp lệ
Giữ nguyên checkpoint hiện có.
Checkpoint: ['checkpoint-3500', 'checkpoint-3600', 'checkpoint-3663']
DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 15620
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords'],
        num_rows: 1953
    })
})


## Tiền xử lý long-context

Toàn bộ tài liệu được xét khi chọn thông tin:

```text
Văn bản dài
→ tách câu
→ chunk theo câu, có chồng lấn
→ chấm điểm bằng từ khóa + độ nổi bật + vị trí
→ MMR giảm chọn các chunk trùng nhau
→ sắp xếp Top-K chunk theo thứ tự gốc
→ ViT5 nhận tối đa 1.024 token
```


In [6]:
# ============================================================
# 5. LOAD VIT5 VÀ CẤU HÌNH LONG-CONTEXT
# ============================================================

import ast
import json
import re
from collections import Counter

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

# ViT5 vẫn dùng full attention; 1.024 token tốn VRAM hơn 512.
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 256
MAX_KEYWORD_TARGET_LENGTH = 64

# Tạo chunk theo câu trên toàn văn.
CHUNK_TOKEN_LENGTH = 256
CHUNK_OVERLAP_SENTENCES = 1
MAX_SELECTED_CHUNKS = 4

# Khoảng 20% mẫu bổ sung nhiệm vụ sinh từ khóa.
KEYWORD_TASK_EVERY = 4

# Trọng số chọn chunk.
KEYWORD_SCORE_WEIGHT = 0.45
SALIENCE_SCORE_WEIGHT = 0.35
POSITION_SCORE_WEIGHT = 0.20
MMR_REDUNDANCY_WEIGHT = 0.20

print("Đang load tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    str(LOCAL_MODEL),
    use_fast=False,
    local_files_only=True,
)

print("Đang load model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    str(LOCAL_MODEL),
    local_files_only=True,
)

model.config.use_cache = False

print(
    "Số tham số:",
    f"{model.num_parameters():,}",
)


VIETNAMESE_STOPWORDS = {
    "và", "là", "của", "có", "cho", "trong", "một",
    "những", "các", "được", "với", "để", "khi", "đã",
    "này", "đó", "từ", "theo", "về", "trên", "tại",
    "như", "do", "thì", "mà", "hay", "hoặc", "cũng",
    "không", "sẽ", "đang", "bị", "ra", "đến", "sau",
    "trước", "nhiều", "người", "việc", "lại", "nên",
}


def normalize_text(value) -> str:
    if value is None:
        return ""

    if isinstance(value, list):
        value = " ".join(
            str(item)
            for item in value
        )

    return " ".join(
        str(value)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .split()
    ).strip()


def word_tokens(text: str) -> list[str]:
    return re.findall(
        r"[0-9A-Za-zÀ-ỹĐđ]+",
        normalize_text(text).lower(),
    )


def split_sentences(text: str) -> list[str]:
    text = str(text).replace(
        "\r\n",
        "\n",
    )

    parts = re.split(
        r"(?<=[.!?…])\s+|\n+",
        text,
    )

    sentences = [
        normalize_text(part)
        for part in parts
        if normalize_text(part)
    ]

    return sentences or [
        normalize_text(text)
    ]


def derive_keywords(
    document: str,
    top_k: int = 8,
) -> list[str]:
    words = [
        word
        for word in word_tokens(document)
        if len(word) >= 3
        and word not in VIETNAMESE_STOPWORDS
        and not word.isdigit()
    ]

    counts = Counter(words)

    return [
        word
        for word, _ in counts.most_common(top_k)
    ]


def normalize_keywords(
    value,
    document: str,
    top_k: int = 10,
) -> list[str]:
    raw_items = []

    if isinstance(value, list):
        raw_items = value

    elif isinstance(value, str):
        stripped = value.strip()

        if stripped:
            parsed = None

            if (
                stripped.startswith("[")
                and stripped.endswith("]")
            ):
                for parser in [
                    json.loads,
                    ast.literal_eval,
                ]:
                    try:
                        parsed = parser(stripped)
                        break
                    except Exception:
                        pass

            if isinstance(parsed, list):
                raw_items = parsed
            else:
                raw_items = re.split(
                    r"[;,\n|]+",
                    stripped,
                )

    keywords = []
    seen = set()

    for item in raw_items:
        keyword = normalize_text(item)

        if not keyword:
            continue

        normalized = keyword.lower()

        if normalized in seen:
            continue

        seen.add(normalized)
        keywords.append(keyword)

        if len(keywords) >= top_k:
            break

    if not keywords:
        keywords = derive_keywords(
            document,
            top_k=min(top_k, 8),
        )

    return keywords


def split_long_sentence_by_tokens(
    sentence: str,
) -> list[str]:
    token_ids = tokenizer.encode(
        sentence,
        add_special_tokens=False,
    )

    if len(token_ids) <= CHUNK_TOKEN_LENGTH:
        return [sentence]

    pieces = []

    for start in range(
        0,
        len(token_ids),
        CHUNK_TOKEN_LENGTH,
    ):
        piece_ids = token_ids[
            start:start + CHUNK_TOKEN_LENGTH
        ]

        piece = normalize_text(
            tokenizer.decode(
                piece_ids,
                skip_special_tokens=True,
            )
        )

        if piece:
            pieces.append(piece)

    return pieces


def build_sentence_chunks(
    document: str,
) -> list[str]:
    expanded_sentences = []

    for sentence in split_sentences(document):
        expanded_sentences.extend(
            split_long_sentence_by_tokens(
                sentence
            )
        )

    chunks = []
    current_sentences = []
    current_tokens = 0

    for sentence in expanded_sentences:
        sentence_tokens = len(
            tokenizer.encode(
                sentence,
                add_special_tokens=False,
            )
        )

        would_overflow = (
            current_sentences
            and current_tokens + sentence_tokens
            > CHUNK_TOKEN_LENGTH
        )

        if would_overflow:
            chunks.append(
                " ".join(current_sentences)
            )

            overlap = (
                current_sentences[
                    -CHUNK_OVERLAP_SENTENCES:
                ]
                if CHUNK_OVERLAP_SENTENCES > 0
                else []
            )

            current_sentences = list(overlap)
            current_tokens = sum(
                len(
                    tokenizer.encode(
                        item,
                        add_special_tokens=False,
                    )
                )
                for item in current_sentences
            )

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(
            " ".join(current_sentences)
        )

    return chunks


def jaccard_similarity(
    text_a: str,
    text_b: str,
) -> float:
    words_a = set(word_tokens(text_a))
    words_b = set(word_tokens(text_b))

    if not words_a or not words_b:
        return 0.0

    return len(words_a & words_b) / len(
        words_a | words_b
    )


def score_and_select_chunks(
    document: str,
    keywords: list[str],
) -> tuple[list[str], list[dict]]:
    chunks = build_sentence_chunks(document)

    if len(chunks) <= MAX_SELECTED_CHUNKS:
        metadata = [
            {
                "index": index,
                "score": 1.0,
            }
            for index in range(len(chunks))
        ]

        return chunks, metadata

    document_words = [
        word
        for word in word_tokens(document)
        if word not in VIETNAMESE_STOPWORDS
    ]

    frequencies = Counter(document_words)
    max_frequency = max(
        frequencies.values(),
        default=1,
    )

    base_scores = []

    for index, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        chunk_words = [
            word
            for word in word_tokens(chunk)
            if word not in VIETNAMESE_STOPWORDS
        ]

        keyword_coverage = (
            sum(
                keyword.lower() in chunk_lower
                for keyword in keywords
            )
            / max(len(keywords), 1)
        )

        salience = (
            sum(
                frequencies[word]
                / max_frequency
                for word in chunk_words
            )
            / max(len(chunk_words), 1)
        )

        position = (
            1.0
            - index
            / max(len(chunks) - 1, 1)
        )

        score = (
            KEYWORD_SCORE_WEIGHT
            * keyword_coverage
            + SALIENCE_SCORE_WEIGHT
            * salience
            + POSITION_SCORE_WEIGHT
            * position
        )

        base_scores.append(score)

    selected_indices = []
    remaining = set(range(len(chunks)))

    while (
        remaining
        and len(selected_indices)
        < MAX_SELECTED_CHUNKS
    ):
        best_index = None
        best_mmr_score = -float("inf")

        for index in remaining:
            redundancy = max(
                (
                    jaccard_similarity(
                        chunks[index],
                        chunks[selected],
                    )
                    for selected
                    in selected_indices
                ),
                default=0.0,
            )

            mmr_score = (
                base_scores[index]
                - MMR_REDUNDANCY_WEIGHT
                * redundancy
            )

            if mmr_score > best_mmr_score:
                best_index = index
                best_mmr_score = mmr_score

        selected_indices.append(best_index)
        remaining.remove(best_index)

    # Giữ lại mạch văn ban đầu.
    selected_indices.sort()

    selected_chunks = [
        chunks[index]
        for index in selected_indices
    ]

    metadata = [
        {
            "index": index,
            "score": round(
                base_scores[index],
                6,
            ),
        }
        for index in selected_indices
    ]

    return selected_chunks, metadata


def build_summary_input(
    document: str,
    keyword_value=None,
) -> tuple[str, list[str], list[dict]]:
    document = normalize_text(document)

    keywords = normalize_keywords(
        keyword_value,
        document=document,
    )

    selected_chunks, chunk_metadata = (
        score_and_select_chunks(
            document,
            keywords,
        )
    )

    selected_context = (
        " ĐOẠN TIẾP: ".join(selected_chunks)
    )

    keyword_text = "; ".join(keywords)

    input_text = (
        "summarize: "
        f"keywords: {keyword_text} "
        f"document: {selected_context}"
    )

    return (
        input_text,
        keywords,
        chunk_metadata,
    )


Đang load tokenizer...
Đang load model...


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 2625.19it/s]

Số tham số: 225,950,976


In [7]:
# ============================================================
# 6. TẠO DỮ LIỆU MULTI-TASK VÀ TOKENIZE
# ============================================================

def preprocess_train_batch(
    examples,
    indices,
):
    documents = examples[TEXT_COLUMN]
    summaries = examples[SUMMARY_COLUMN]

    keyword_values = (
        examples[KEYWORDS_COLUMN]
        if KEYWORDS_COLUMN in examples
        else [None] * len(documents)
    )

    input_texts = []
    target_texts = []

    for local_index, (
        document,
        summary,
        keyword_value,
    ) in enumerate(
        zip(
            documents,
            summaries,
            keyword_values,
        )
    ):
        summary_input, keywords, _ = (
            build_summary_input(
                document,
                keyword_value,
            )
        )

        # Nhiệm vụ chính: tóm tắt.
        input_texts.append(summary_input)
        target_texts.append(
            normalize_text(summary)
        )

        # Nhiệm vụ phụ: sinh từ khóa (~20%).
        global_index = indices[local_index]

        if (
            KEYWORD_TASK_EVERY > 0
            and global_index
            % KEYWORD_TASK_EVERY
            == 0
        ):
            keyword_context = (
                summary_input.split(
                    "document:",
                    maxsplit=1,
                )[-1]
            )

            input_texts.append(
                "keywords: document: "
                + keyword_context
            )

            target_texts.append(
                "; ".join(keywords)
            )

    model_inputs = tokenizer(
        input_texts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=target_texts,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = (
        labels["input_ids"]
    )

    return model_inputs


def preprocess_summary_batch(examples):
    documents = examples[TEXT_COLUMN]
    summaries = examples[SUMMARY_COLUMN]

    keyword_values = (
        examples[KEYWORDS_COLUMN]
        if KEYWORDS_COLUMN in examples
        else [None] * len(documents)
    )

    input_texts = []

    for document, keyword_value in zip(
        documents,
        keyword_values,
    ):
        summary_input, _, _ = (
            build_summary_input(
                document,
                keyword_value,
            )
        )

        input_texts.append(summary_input)

    model_inputs = tokenizer(
        input_texts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=[
            normalize_text(summary)
            for summary in summaries
        ],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = (
        labels["input_ids"]
    )

    return model_inputs


print("Đang tạo train multi-task...")

tokenized_train = (
    dataset_for_training["train"].map(
        preprocess_train_batch,
        batched=True,
        with_indices=True,
        remove_columns=dataset_for_training[
            "train"
        ].column_names,
        desc="Train: long-context + keyword task",
    )
)

print("Đang tokenize validation/test...")

tokenized_validation = (
    dataset_for_training[
        "validation"
    ].map(
        preprocess_summary_batch,
        batched=True,
        remove_columns=dataset_for_training[
            "validation"
        ].column_names,
        desc="Validation: summary task",
    )
)

tokenized_test = (
    dataset_for_training["test"].map(
        preprocess_summary_batch,
        batched=True,
        remove_columns=dataset_for_training[
            "test"
        ].column_names,
        desc="Test: summary task",
    )
)

tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "validation": tokenized_validation,
    "test": tokenized_test,
})

print(tokenized_dataset)
print(
    "Train gốc:",
    len(dataset_for_training["train"]),
)
print(
    "Train sau multi-task:",
    len(tokenized_dataset["train"]),
)


Đang tạo train multi-task...
Đang tokenize validation/test...
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 19525
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1952
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1953
    })
})
Train gốc: 15620
Train sau multi-task: 19525


In [8]:
# ============================================================
# 7. METRIC AN TOÀN, KHÔNG LỖI DECODE TOKEN ÂM
# ============================================================

import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")


def sanitize_token_ids(values):
    values = np.asarray(values)

    if values.ndim == 3:
        values = np.argmax(
            values,
            axis=-1,
        )

    if np.issubdtype(
        values.dtype,
        np.floating,
    ):
        values = np.nan_to_num(
            values,
            nan=tokenizer.pad_token_id,
            posinf=tokenizer.pad_token_id,
            neginf=tokenizer.pad_token_id,
        )

    values = values.astype(np.int64)

    valid_mask = (
        (values >= 0)
        & (values < len(tokenizer))
    )

    return np.where(
        valid_mask,
        values,
        tokenizer.pad_token_id,
    )


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = sanitize_token_ids(
        predictions
    )

    labels = sanitize_token_ids(labels)

    decoded_predictions = (
        tokenizer.batch_decode(
            predictions,
            skip_special_tokens=True,
        )
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
    )

    decoded_predictions = [
        normalize_text(item)
        for item in decoded_predictions
    ]

    decoded_labels = [
        normalize_text(item)
        for item in decoded_labels
    ]

    result = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=False,
    )

    result["gen_len"] = float(
        np.mean([
            np.count_nonzero(
                prediction
                != tokenizer.pad_token_id
            )
            for prediction
            in predictions
        ])
    )

    return {
        key: round(float(value), 4)
        for key, value in result.items()
    }


## Huấn luyện an toàn

Trong khi train **không chạy validation tự động**. Trainer lưu checkpoint đầy đủ trước mỗi 100 optimizer step. Vì vậy lỗi đánh giá không làm mất nhiều giờ huấn luyện.


In [9]:
# ============================================================
# 8. TRAINER: ƯU TIÊN CHECKPOINT AN TOÀN
# ============================================================

import inspect

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# 1.024 token cần batch nhỏ hơn.
# Batch hiệu dụng = 1 × 15 = 15.
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 3e-5
NUM_TRAIN_EPOCHS = 3

SAVE_STEPS = 100
EVAL_STEPS = 200
LOGGING_STEPS = 20
SAVE_TOTAL_LIMIT = 3

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)


def create_training_arguments():
    argument_names = inspect.signature(
        Seq2SeqTrainingArguments.__init__
    ).parameters

    use_bf16 = (
        torch.cuda.is_available()
        and hasattr(
            torch.cuda,
            "is_bf16_supported",
        )
        and torch.cuda.is_bf16_supported()
    )

    kwargs = {
        "output_dir": str(CHECKPOINT_DIR),

        "num_train_epochs":
            NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "linear",

        "per_device_train_batch_size":
            TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size":
            EVAL_BATCH_SIZE,
        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "gradient_checkpointing": True,

        "fp16": (
            torch.cuda.is_available()
            and not use_bf16
        ),
        "bf16": use_bf16,

        "logging_strategy": "steps",
        "logging_steps": LOGGING_STEPS,
        "logging_first_step": True,

        # save không bị chặn bởi compute_metrics.
        "save_strategy": "steps",
        "save_steps": SAVE_STEPS,
        "save_total_limit":
            SAVE_TOTAL_LIMIT,
        "save_only_model": False,
        "eval_steps": EVAL_STEPS,

        "predict_with_generate": True,
        "generation_max_length":
            MAX_TARGET_LENGTH,
        "generation_num_beams": 4,

        "load_best_model_at_end": False,

        "report_to": "none",
        "push_to_hub": False,

        "seed": 42,
        "data_seed": 42,

        "dataloader_num_workers": 0,
        "dataloader_pin_memory": False,
    }

    if "eval_strategy" in argument_names:
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs[
            "evaluation_strategy"
        ] = "steps"

    return Seq2SeqTrainingArguments(
        **kwargs
    )


training_args = create_training_arguments()

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset":
        tokenized_dataset["train"],
    "eval_dataset":
        tokenized_dataset["validation"].select(range(50)),
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_parameters = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs[
        "processing_class"
    ] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(
    **trainer_kwargs
)

print(training_args)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=False,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=200,
eval_strategy=steps,
eval_use_gather_o

In [10]:
# ============================================================
# 9. TÌM CHECKPOINT HỢP LỆ MỚI NHẤT
# ============================================================

def find_latest_valid_checkpoint(
    checkpoint_root: Path,
):
    valid_checkpoints = []

    print(
        "Đang tìm checkpoint tại:",
        checkpoint_root.resolve(),
    )

    for path in checkpoint_root.glob(
        "checkpoint-*"
    ):
        if not path.is_dir():
            continue

        try:
            step = int(
                path.name.rsplit(
                    "-",
                    1,
                )[-1]
            )
        except ValueError:
            continue

        model_files = [
            "model.safetensors",
            "model.safetensors.index.json",
            "pytorch_model.bin",
            "pytorch_model.bin.index.json",
        ]

        required_state_files = [
            "trainer_state.json",
            "optimizer.pt",
            "scheduler.pt",
        ]

        has_model = any(
            (path / file_name).exists()
            for file_name in model_files
        )

        missing_state_files = [
            file_name
            for file_name
            in required_state_files
            if not (
                path / file_name
            ).exists()
        ]

        print(
            path.name,
            "| model:",
            has_model,
            "| thiếu:",
            missing_state_files,
        )

        if (
            has_model
            and not missing_state_files
        ):
            valid_checkpoints.append(
                (step, path)
            )

    if not valid_checkpoints:
        return None

    valid_checkpoints.sort(
        key=lambda item: item[0]
    )

    return str(
        valid_checkpoints[-1][1]
    )


latest_checkpoint = (
    find_latest_valid_checkpoint(
        CHECKPOINT_DIR
    )
)

if latest_checkpoint:
    print(
        "Sẽ tiếp tục từ:",
        latest_checkpoint,
    )
else:
    print(
        "Không có checkpoint hợp lệ; "
        "train từ model gốc."
    )


Đang tìm checkpoint tại: D:\homework\BTL_NLP\training\vit5-base-hlk-0001-checkpoints
checkpoint-3500 | model: True | thiếu: []
checkpoint-3600 | model: True | thiếu: []
checkpoint-3663 | model: True | thiếu: []
Sẽ tiếp tục từ: D:\homework\BTL_NLP\training\vit5-base-hlk-0001-checkpoints\checkpoint-3663


In [11]:
# ============================================================
# 10. TRAIN
# ============================================================


train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

print("Train hoàn tất.")
print(train_result.metrics)


[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	eval_steps: 200 (from args) != 500 (from trainer_state.json)
	per_device_train_batch_size: 2 (from args) != 1 (from trainer_state.json)


Step,Training Loss,Validation Loss


Train hoàn tất.
{'train_runtime': 0.005, 'train_samples_per_second': 11740483.456, 'train_steps_per_second': 734193.613, 'total_flos': 6.581581675794432e+16, 'train_loss': 0.0, 'epoch': 3.0}


In [12]:
# ============================================================
# 11. LƯU MODEL CUỐI NGAY SAU TRAIN
# ============================================================

model.config.use_cache = True

trainer.save_model(
    str(FINAL_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(FINAL_MODEL_DIR)
)

trainer.save_state()

trainer.log_metrics(
    "train",
    train_result.metrics,
)

trainer.save_metrics(
    "train",
    train_result.metrics,
)

pipeline_config = {
    "max_input_length":
        MAX_INPUT_LENGTH,
    "max_target_length":
        MAX_TARGET_LENGTH,
    "chunk_token_length":
        CHUNK_TOKEN_LENGTH,
    "chunk_overlap_sentences":
        CHUNK_OVERLAP_SENTENCES,
    "max_selected_chunks":
        MAX_SELECTED_CHUNKS,
    "keyword_task_every":
        KEYWORD_TASK_EVERY,
}

with (
    FINAL_MODEL_DIR
    / "hlk_pipeline_config.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        pipeline_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Đã lưu model:",
    FINAL_MODEL_DIR,
)


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]

***** train metrics *****
  epoch                    =          3.0
  total_flos               =   61295755GF
  train_loss               =          0.0
  train_runtime            =   0:00:00.00
  train_samples_per_second = 11740483.456
  train_steps_per_second   =   734193.613
Đã lưu model: D:\homework\BTL_NLP\models\ViT5-base-HLK-0001


In [13]:
# ============================================================
# 12. ĐÁNH GIÁ SAU KHI MODEL ĐÃ ĐƯỢC LƯU
# ============================================================

# Đặt None để đánh giá toàn bộ.
# Nên thử 50–200 mẫu trước vì beam search khá chậm.
EVALUATION_MAX_SAMPLES = 5

evaluation_dataset = (
    tokenized_dataset["test"]
)

original_test_dataset = (
    dataset_for_training["test"]
)

if EVALUATION_MAX_SAMPLES is not None:
    limit = min(
        EVALUATION_MAX_SAMPLES,
        len(evaluation_dataset),
    )

    evaluation_dataset = (
        evaluation_dataset.select(
            range(limit)
        )
    )

    original_test_dataset = (
        original_test_dataset.select(
            range(limit)
        )
    )

print(
    "Số mẫu đánh giá:",
    len(evaluation_dataset),
)

test_result = trainer.predict(
    evaluation_dataset,
    metric_key_prefix="test",
)

trainer.log_metrics(
    "test",
    test_result.metrics,
)

trainer.save_metrics(
    "test",
    test_result.metrics,
)

print(test_result.metrics)


Số mẫu đánh giá: 5


***** test metrics *****
  test_gen_len            =      232.0
  test_loss               =     1.1225
  test_rouge1             =     0.6846
  test_rouge2             =     0.3516
  test_rougeL             =     0.3967
  test_rougeLsum          =     0.3953
  test_runtime            = 0:00:27.93
  test_samples_per_second =      0.179
  test_steps_per_second   =      0.107
{'test_loss': 1.1225390434265137, 'test_rouge1': 0.6846, 'test_rouge2': 0.3516, 'test_rougeL': 0.3967, 'test_rougeLsum': 0.3953, 'test_gen_len': 232.0, 'test_runtime': 27.9369, 'test_samples_per_second': 0.179, 'test_steps_per_second': 0.107}


## Inference: sinh nhiều ứng viên và factuality reranking

Mỗi bài được xử lý bằng cùng cơ chế long-context như lúc train. ViT5 sinh nhiều ứng viên; bộ reranker ưu tiên:

- từ ngữ có căn cứ trong nguồn;
- bao phủ từ khóa;
- số liệu và thực thể xuất hiện trong nguồn;
- độ dài hợp lý;
- ít lặp.


In [14]:
# ============================================================
# 13. BATCH TEST + FACTUALITY RERANKING
# Mỗi bài được lưu ngay thành một file TXT.
# ============================================================

import time
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

MODEL_DIR = FINAL_MODEL_DIR
TEST_FILE = LOCAL_DATASET / "test.jsonl"

OUTPUT_DIR = (
    RESULT_DIR
    / "test_predictions_txt"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

tokenizer_test = (
    AutoTokenizer.from_pretrained(
        str(MODEL_DIR),
        use_fast=False,
        local_files_only=True,
    )
)

model_test = (
    AutoModelForSeq2SeqLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
    )
)

model_test.to(device)
model_test.eval()

NUM_BEAMS = 4
NUM_RETURN_SEQUENCES = 4
MAX_NEW_TOKENS = 300
MIN_NEW_TOKENS = 30

SKIP_EXISTING = True
MAX_SAMPLES = 300

# True: dùng keywords có sẵn trong JSONL.
# False: tự trích xuất keywords từ document.
USE_DATASET_KEYWORDS = True


def extract_capitalized_entities(
    text: str,
) -> set[str]:
    entities = re.findall(
        r"\b(?:[A-ZÀ-ỸĐ][\wÀ-ỹĐđ.-]*"
        r"(?:\s+[A-ZÀ-ỸĐ][\wÀ-ỹĐđ.-]*)+)\b",
        text,
    )

    return {
        normalize_text(entity).lower()
        for entity in entities
        if normalize_text(entity)
    }


def extract_numbers(text: str) -> set[str]:
    return set(
        re.findall(
            r"\b\d+(?:[.,]\d+)*\b",
            text,
        )
    )


def repetition_ratio(
    text: str,
    n: int = 3,
) -> float:
    words = word_tokens(text)

    if len(words) < n:
        return 0.0

    ngrams = [
        tuple(words[index:index + n])
        for index in range(
            len(words) - n + 1
        )
    ]

    return (
        1.0
        - len(set(ngrams))
        / max(len(ngrams), 1)
    )


def candidate_score(
    summary: str,
    document: str,
    keywords: list[str],
) -> dict:
    summary_words = set(
        word_tokens(summary)
    )

    document_words = set(
        word_tokens(document)
    )

    source_precision = (
        len(
            summary_words
            & document_words
        )
        / max(len(summary_words), 1)
    )

    summary_lower = summary.lower()

    keyword_coverage = (
        sum(
            keyword.lower()
            in summary_lower
            for keyword in keywords
        )
        / max(len(keywords), 1)
    )

    summary_numbers = extract_numbers(
        summary
    )

    document_numbers = extract_numbers(
        document
    )

    number_consistency = (
        1.0
        if not summary_numbers
        else len(
            summary_numbers
            & document_numbers
        )
        / len(summary_numbers)
    )

    summary_entities = (
        extract_capitalized_entities(
            summary
        )
    )

    document_entities = (
        extract_capitalized_entities(
            document
        )
    )

    entity_consistency = (
        1.0
        if not summary_entities
        else len(
            summary_entities
            & document_entities
        )
        / len(summary_entities)
    )

    input_length = max(
        len(word_tokens(document)),
        1,
    )

    output_length = len(
        word_tokens(summary)
    )

    compression_ratio = (
        output_length / input_length
    )

    # Tốt nhất quanh 10–25% độ dài đầu vào.
    length_score = max(
        0.0,
        1.0
        - abs(
            compression_ratio - 0.17
        )
        / 0.17,
    )

    repeat_ratio = repetition_ratio(
        summary
    )

    total_score = (
        0.35 * source_precision
        + 0.25 * keyword_coverage
        + 0.15 * number_consistency
        + 0.10 * entity_consistency
        + 0.10 * length_score
        + 0.05 * (1.0 - repeat_ratio)
    )

    return {
        "score": total_score,
        "source_precision":
            source_precision,
        "keyword_coverage":
            keyword_coverage,
        "number_consistency":
            number_consistency,
        "entity_consistency":
            entity_consistency,
        "length_score": length_score,
        "repetition_ratio":
            repeat_ratio,
    }


@torch.inference_mode()
def summarize_with_reranking(
    document: str,
    keyword_value=None,
):
    if not USE_DATASET_KEYWORDS:
        keyword_value = None

    input_text, keywords, chunk_metadata = (
        build_summary_input(
            document,
            keyword_value,
        )
    )

    inputs = tokenizer_test(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        padding=False,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    output_ids = model_test.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens=MIN_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        num_return_sequences=
            NUM_RETURN_SEQUENCES,
        do_sample=False,
        early_stopping=False,
        length_penalty=1.2,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
    )

    candidates = tokenizer_test.batch_decode(
        output_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    unique_candidates = []
    seen = set()

    for candidate in candidates:
        candidate = normalize_text(
            candidate
        )

        if (
            candidate
            and candidate not in seen
        ):
            seen.add(candidate)
            unique_candidates.append(
                candidate
            )

    ranked_candidates = []

    for candidate in unique_candidates:
        details = candidate_score(
            candidate,
            document,
            keywords,
        )

        ranked_candidates.append({
            "summary": candidate,
            **details,
        })

    ranked_candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    best = (
        ranked_candidates[0]
        if ranked_candidates
        else {
            "summary": "",
            "score": 0.0,
        }
    )

    return {
        "final_summary":
            best["summary"],
        "best_score":
            best["score"],
        "keywords": keywords,
        "selected_chunks":
            chunk_metadata,
        "candidates":
            ranked_candidates,
        "input_tokens":
            int(
                inputs[
                    "input_ids"
                ].shape[1]
            ),
    }


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(
                    json.loads(line)
                )
            except json.JSONDecodeError as error:
                print(
                    "Bỏ qua dòng",
                    line_number,
                    error,
                )

    return records


def safe_filename(
    value: str,
    max_length: int = 70,
) -> str:
    value = normalize_text(value)
    value = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        value,
    )
    value = re.sub(
        r"\s+",
        "_",
        value,
    )
    value = value.strip("._ ")

    return (
        value[:max_length]
        if value
        else "untitled"
    )


def save_result_txt(
    output_path: Path,
    index: int,
    guid: str,
    title: str,
    article: str,
    reference: str,
    generated: str,
    elapsed_seconds: float,
    rerank_score: float,
    keywords: list[str],
):
    content = (
        "INDEX:\n"
        f"{index}\n"
        "GUID:\n"
        f"{guid}\n"
        "TITLE:\n"
        f"{title}\n"
        "VĂN BẢN GỐC:\n"
        f"{article}\n"
        "TÓM TẮT THAM CHIẾU:\n"
        f"{reference}\n"
        "TÓM TẮT DO MODEL SINH:\n"
        f"{generated}\n"
        "THỜI GIAN TÓM TẮT:\n"
        f"{elapsed_seconds:.4f} giây\n"
    )

    temporary_path = (
        output_path.with_suffix(
            ".tmp"
        )
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(content)
        file.flush()

    temporary_path.replace(
        output_path
    )


records = load_jsonl(TEST_FILE)

if MAX_SAMPLES is not None:
    records = records[:MAX_SAMPLES]

success_count = 0
skip_count = 0
error_count = 0

for index, record in enumerate(
    tqdm(
        records,
        desc="HLK-ViT5 test",
    )
):
    guid = normalize_text(
        record.get(
            "guid",
            record.get("id", index),
        )
    )

    title = normalize_text(
        record.get("title", "")
    )

    article = normalize_text(
        record.get(
            TEXT_COLUMN,
            record.get("text", ""),
        )
    )

    reference = normalize_text(
        record.get(
            SUMMARY_COLUMN,
            "",
        )
    )

    keyword_value = record.get(
        KEYWORDS_COLUMN
    )

    output_path = (
        OUTPUT_DIR
        / (
            f"{index:05d}"
            f"_guid-{safe_filename(guid, 25)}"
            f"_{safe_filename(title)}.txt"
        )
    )

    if (
        SKIP_EXISTING
        and output_path.exists()
    ):
        skip_count += 1
        continue

    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        started_at = time.perf_counter()

        result = summarize_with_reranking(
            article,
            keyword_value,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = (
            time.perf_counter()
            - started_at
        )

        save_result_txt(
            output_path=output_path,
            index=index,
            guid=guid,
            title=title,
            article=article,
            reference=reference,
            generated=result[
                "final_summary"
            ],
            elapsed_seconds=elapsed,
            rerank_score=result[
                "best_score"
            ],
            keywords=result["keywords"],
        )

        success_count += 1

    except Exception as error:
        error_count += 1
        print(
            f"Lỗi bài {index}:",
            repr(error),
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nHoàn tất.")
print("Tạo mới :", success_count)
print("Bỏ qua  :", skip_count)
print("Bị lỗi  :", error_count)
print("Kết quả :", OUTPUT_DIR.resolve())


HLK-ViT5 test: 100%|██████████| 300/300 [19:38<00:00,  3.93s/it]


Hoàn tất.
Tạo mới : 219
Bỏ qua  : 81
Bị lỗi  : 0
Kết quả : D:\homework\BTL_NLP\results\vit5-base-HLK-0001\test_predictions_txt


In [15]:
# ============================================================
# 14. CHECKPOINT KHẨN CẤP
# Chỉ chạy khi trainer.train() vừa lỗi nhưng kernel còn sống.
# ============================================================

import inspect

print(
    "Global step:",
    trainer.state.global_step,
)

save_function = (
    trainer._save_checkpoint
)

parameters = inspect.signature(
    save_function
).parameters

kwargs = {}

if "model" in parameters:
    kwargs["model"] = trainer.model

if "trial" in parameters:
    kwargs["trial"] = None

save_function(**kwargs)

print(
    "Đã lưu checkpoint khẩn cấp tại:",
    (
        Path(trainer.args.output_dir)
        / (
            "checkpoint-"
            + str(
                trainer.state.global_step
            )
        )
    ),
)


Global step: 3663


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.23s/it]


Đã lưu checkpoint khẩn cấp tại: D:\homework\BTL_NLP\training\vit5-base-hlk-0001-checkpoints\checkpoint-3663


In [ ]:
# Kiểm tra xem CUDA có khả dụng và in thông tin môi trường.

import sys
import torch

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("Torch path:", torch.__file__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: d:\homework\BTL_NLP\.venv\Scripts\python.exe
Torch version: 2.11.0+cu128
Torch path: d:\homework\BTL_NLP\.venv\lib\site-packages\torch\__init__.py
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


: 